# CFPB Consumer Complaints: Data Loading + Exploratory Data Analysis (EDA)

This notebook starts the project with the official CFPB consumer complaints data.
Its role is descriptive only: data loading, quality checks, and exploratory visuals for modeling readiness.

## Notebook Goals
- Load and validate the sampled complaints data file
- Inspect schema, data types, and missingness patterns
- Explore `Company response to consumer` to understand potential target design
- Check text feature availability (`Consumer complaint narrative`)
- Identify risks/assumptions before preprocessing

**Boundary:** final target construction and training feature creation are done in `02_preprocessing.ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path('../data/processed')

# Load the sampled data (created by src/sample_cfpb_data.py)
sample_file = data_dir / 'complaints_sample.csv'

if not sample_file.exists():
    raise FileNotFoundError(
        f'{sample_file} not found. Run: python src/sample_cfpb_data.py --sample-size 5000'
    )

df = pd.read_csv(sample_file)
print(f"Loaded {len(df):,} rows from {sample_file.name}")
print(f"Shape: {df.shape}")
df.head(2)

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
})
summary.sort_values('missing_pct', ascending=False).head(20)

In [ ]:
response_col = 'Company response to consumer'
if response_col in df.columns:
    response_counts = df[response_col].fillna('Missing').value_counts(dropna=False)
    display(response_counts.head(20))

    # EDA-only proxy check to understand likely class balance.
    # Final target definition used for modeling is created in notebook 02.
    money_keywords = ['monetary', 'refund', 'credit', 'restitution', 'compensation', 'relief']
    response_text = df[response_col].fillna('').str.lower()
    eda_monetary_proxy = response_text.str.contains('|'.join(money_keywords), regex=True).astype(int)

    print('EDA proxy class percentages (%):')
    print((eda_monetary_proxy.value_counts(normalize=True).sort_index() * 100).round(2))
else:
    print(f'{response_col} not found. Available columns:')
    print(list(df.columns[:50]))

In [ ]:
narrative_col = 'Consumer complaint narrative'
if narrative_col in df.columns:
    narrative_availability = df[narrative_col].notna().mean() * 100
    print(f'Narrative availability: {narrative_availability:.2f}%')
else:
    print(f'{narrative_col} not found.')

## Additional EDA: Visual Diagnostics

These visuals help us quickly spot class imbalance, concentration effects, and missing-data risk before preprocessing/modeling.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

def plot_top_counts(df, col, top_n=10, title=None, rotate_xticks=False):
    if col not in df.columns:
        print(f"Column not found: {col}")
        return
    counts = df[col].fillna('Missing').value_counts().head(top_n)
    ax = counts.sort_values().plot(kind='barh')
    ax.set_title(title or f'Top {top_n} values in {col}')
    ax.set_xlabel('Count')
    ax.set_ylabel(col)
    if rotate_xticks:
        plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Missingness profile for top columns by missing percentage
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
top_missing = missing_pct.head(15)

ax = top_missing.sort_values().plot(kind='barh')
ax.set_title('Top 15 Columns by Missing Percentage')
ax.set_xlabel('Missing (%)')
ax.set_ylabel('Column')
plt.tight_layout()
plt.show()

In [ ]:
# EDA-only view: monetary-relief proxy distribution
response_col = 'Company response to consumer'
if response_col in df.columns:
    money_keywords = ['monetary', 'refund', 'credit', 'restitution', 'compensation', 'relief']
    response_text = df[response_col].fillna('').str.lower()
    eda_monetary_proxy = response_text.str.contains('|'.join(money_keywords), regex=True).astype(int)

    proxy_counts = eda_monetary_proxy.value_counts().sort_index()
    label_map = {0: 'No monetary relief (proxy)', 1: 'Monetary relief (proxy)'}
    plot_labels = [label_map.get(i, str(i)) for i in proxy_counts.index]

    ax = proxy_counts.plot(kind='bar', color=['#4C78A8', '#F58518'])
    ax.set_title('EDA Proxy Distribution (for target design only)')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.set_xticklabels(plot_labels, rotation=0)
    plt.tight_layout()
    plt.show()

    print('EDA proxy percentages (%):')
    print((proxy_counts / proxy_counts.sum() * 100).round(2))
    print('\nNote: final modeling target is defined in 02_preprocessing.ipynb.')
else:
    print(f"Column not found: {response_col}")

In [ ]:
# Core categorical distributions
plot_top_counts(df, 'Product', top_n=12, title='Top Products')
plot_top_counts(df, 'Issue', top_n=12, title='Top Issues')
plot_top_counts(df, 'Company', top_n=12, title='Top Companies')
plot_top_counts(df, 'State', top_n=12, title='Top States')

# Submission channel and company response patterns (if present)
plot_top_counts(df, 'Submitted via', top_n=10, title='Submission Channels')
plot_top_counts(df, 'Company response to consumer', top_n=10, title='Company Response Categories')